In [ ]:
"""
SEED Dataset: EEGNet + 90%训练 + 10%验证
========================================
合并train+val后，再分出10%作为验证集监控。
随机种子已固定，结果可复现。
"""

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# =================== 固定随机种子（可复现） ===================
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("✓ 随机种子已固定 (seed=42)，结果可复现")

# =================== 配置 ===================
DATA_DIR = '/mnt/dataset0/Jity/EEG-Conformer-main/EEG-Conformer-main/data/SEED'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCHS = 150
LR = 0.001
PATIENCE = 25
VAL_SPLIT = 0.1  # 留10%做验证

print(f"使用设备: {DEVICE}")

# =================== 数据加载 ===================

def load_h5(filepath):
    data_dict = {}
    with h5py.File(filepath, 'r') as f:
        for key in f.keys():
            data_dict[key] = f[key][()]
    return data_dict

print("[步骤1] 加载数据...")
train_data = load_h5(f'{DATA_DIR}/train.h5')
val_data = load_h5(f'{DATA_DIR}/val.h5')
test_data = load_h5(f'{DATA_DIR}/test_x_only.h5')

# 合并 train + val
X_combined = np.concatenate([train_data['X'], val_data['X']], axis=0)
y_combined = np.concatenate([train_data['y'], val_data['y']], axis=0)
X_test = test_data['X']

print(f"  合并后总数据: {X_combined.shape}")

# 分出10%做验证（随机分，保持标签比例）
X_train, X_val, y_train, y_val = train_test_split(
    X_combined, y_combined, test_size=VAL_SPLIT,
    random_state=42, stratify=y_combined
)

print(f"  训练集: {X_train.shape}")
print(f"  验证集: {X_val.shape}")
print(f"  测试集: {X_test.shape}")

print("[步骤2] 转为Tensor...")
X_train_t = torch.FloatTensor(X_train).unsqueeze(1)
y_train_t = torch.LongTensor(y_train)
X_val_t = torch.FloatTensor(X_val).unsqueeze(1)
y_val_t = torch.LongTensor(y_val)
X_test_t = torch.FloatTensor(X_test).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)

# =================== EEGNet ===================

class EEGNet(nn.Module):
    def __init__(self, num_classes=3, channels=62, samples=400, 
                 dropout_rate=0.5, kernel_length=64, F1=8, D=2):
        super(EEGNet, self).__init__()
        self.F1 = F1
        self.D = D
        self.F2 = F1 * D
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kernel_length), padding=(0, kernel_length//2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, self.F2, (channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(self.F2),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout_rate)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(self.F2, self.F2, (1, 16), padding=(0, 8), groups=self.F2, bias=False),
            nn.Conv2d(self.F2, self.F2, 1, bias=False),
            nn.BatchNorm2d(self.F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout_rate)
        )
        
        self.flatten_dim = self.F2 * 12
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_dim, num_classes)
        )
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.classifier(x)
        return x

# =================== 训练函数 ===================

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    return total_loss / len(loader), correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
    return total_loss / len(loader), correct / total, np.array(all_preds), np.array(all_labels)

# =================== 主训练流程 ===================

print("\n[步骤3] 构建 EEGNet...")
model = EEGNet(num_classes=3, channels=62, samples=400, 
               dropout_rate=0.5, kernel_length=64, F1=8, D=2).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"模型总参数量: {total_params:,}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=8, verbose=True)

best_val_acc = 0
best_epoch = 0
no_improve = 0

print(f"\n[步骤4] 开始训练 ({EPOCHS}轮)...")
print("-" * 60)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, val_preds, val_labels = eval_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step(val_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        # 深拷贝！保存真正的最好模型
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        
    if no_improve >= PATIENCE:
        print(f"\n⏹️ Early stopping (best={best_val_acc:.4f} at epoch {best_epoch})")
        break

model.load_state_dict(best_model_state)

# =================== 最终评估 ===================

print(f"\n[步骤5] 最终评估 (Epoch {best_epoch})...")
print("-" * 60)
_, _, val_preds, val_labels = eval_epoch(model, val_loader, criterion, DEVICE)
print(f"验证集准确率: {accuracy_score(val_labels, val_preds):.4f}")
print(classification_report(val_labels, val_preds, target_names=['negative', 'neutral', 'positive']))
print("混淆矩阵:")
print(confusion_matrix(val_labels, val_preds))

# =================== 预测 ===================

print("\n[步骤6] 生成测试集预测...")
model.eval()
with torch.no_grad():
    X_test_device = X_test_t.to(DEVICE)
    test_outputs = model(X_test_device)
    _, test_pred = torch.max(test_outputs, 1)
    test_pred = test_pred.cpu().numpy()

output_file = 'seed_split_predictions.txt'
np.savetxt(output_file, test_pred, fmt='%d')
print(f"\n✅ 已保存: {output_file}")
print(f"   negative={np.sum(test_pred==0)}, neutral={np.sum(test_pred==1)}, positive={np.sum(test_pred==2)}")

print("\n" + "=" * 60)
print("完成！")
print("=" * 60)

✓ 随机种子已固定 (seed=42)，结果可复现
使用设备: cuda
[步骤1] 加载数据...
  合并后总数据: (1350, 62, 400)
  训练集: (1215, 62, 400)
  验证集: (135, 62, 400)
  测试集: (450, 62, 400)
[步骤2] 转为Tensor...

[步骤3] 构建 EEGNet...
模型总参数量: 2,675

[步骤4] 开始训练 (150轮)...
------------------------------------------------------------
Epoch   1/150 | Train: 0.3374 | Val: 0.3556
Epoch   5/150 | Train: 0.3942 | Val: 0.3333
Epoch  10/150 | Train: 0.4634 | Val: 0.4667
Epoch  15/150 | Train: 0.4798 | Val: 0.4963
Epoch  20/150 | Train: 0.5037 | Val: 0.4815
Epoch  25/150 | Train: 0.5383 | Val: 0.4963
Epoch  30/150 | Train: 0.5481 | Val: 0.4889
Epoch 00035: reducing learning rate of group 0 to 5.0000e-04.
Epoch  35/150 | Train: 0.5630 | Val: 0.4519
Epoch  40/150 | Train: 0.5737 | Val: 0.4741
Epoch 00044: reducing learning rate of group 0 to 2.5000e-04.
Epoch  45/150 | Train: 0.5811 | Val: 0.4741
Epoch  50/150 | Train: 0.5934 | Val: 0.4889

⏹️ Early stopping (best=0.5185 at epoch 26)

[步骤5] 最终评估 (Epoch 26)...
------------------------------------------